# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
df = pd.read_csv(DATA_PATH)
print("Using data at:", DATA_PATH)


Using data at: /home/claude/ML-Internship/data/raw/content_refresh_anonymized.csv


## 1. Two paper findings + my methodology questions

### Finding chosen 1: "What Predicts Growth?" (ML Appendix, logistic regression, 71% holdout accuracy)

The paper reports 71% holdout accuracy for a logistic regression separating growing from
declining pages, with content age, days since update, and days visible as the strongest
coefficients.

**My methodology question: what's the base rate, and is the split grouped?** The paper never
states what share of pages are "up" versus "down" next to the 71% number, and accuracy without
a base rate is close to meaningless, 71% on a label that's already 62% one class is 9 points of
real skill, not 71. The paper's own Finding #1 table happens to report the counts this needs
(74.8K up, 45.6K down), the cell below does that arithmetic. Separately, the Methodology page
states "Random Forest (80/20 split), Logistic Regression (80/20 split)" without saying whether
that split is grouped by brand. With 57 brands contributing very unevenly, a plain random 80/20
split risks the same client-memorization problem my own Week 5 model had to guard against with
`GroupShuffleSplit`, section 2 below shows concretely how large that gap can be on data I have
direct access to.

This isn't a claim their number is wrong, the paper may well have grouped the split, it's a
question the disclosed methodology page doesn't answer, and it's the specific piece of context
that would tell a reader how much of that 71% is genuine model skill.

### Finding chosen 2: "Click Capture by Position Tier" (Finding #3, weighted CTR by tier)

The paper reports one weighted CTR per position tier (Top 3: 0.423%, Page 1: 0.339%, down to
Deep: 0.050%), computed as total clicks over total impressions per tier, and explicitly notes
this replaces an earlier per-row average that produced "impossible values above 100%", the exact
CTR-scale trap Discovery A/B caught in my own Week 1 work with the `feedly article` outlier.

**My methodology question: does the pooled tier number hide within-tier heterogeneity by content
type or intent, and what's n per tier?** The paper states elsewhere that the ML appendix uses a
minimum bucket size of n=50 (n=30 for cross-correlations), but Finding #3 is a direct aggregate
comparison, not an ML appendix table, so it's not clear that same floor applies here, or what n
backs each of the five tier numbers. I can't check FlyRank's literal warehouse split by content
type, I don't have access to it, but the cell below reruns the same pooling question on my own
lane's data, and it reproduces the exact heterogeneity problem I'm asking about: the pooled
tier-level pattern is clean and well-supported, but breaking the same tiers out by content type
immediately surfaces cells with single-digit sample sizes. That's a reasonable, concrete question
to ask of any pooled tier average, not a claim that FlyRank's own number is affected.


In [2]:
# Methodology question 1: base rate behind the paper's own 71% holdout accuracy claim
# (counts sourced from the paper's own Finding #1 table: 74.8K up, 45.6K down)
up, down = 74800, 45600
base_rate = up / (up + down)
reported_accuracy = 0.71
print(f"Implied base rate for the majority class ('up'): {base_rate:.3f}")
print(f"Reported holdout accuracy: {reported_accuracy:.2f}")
print(f"Real skill above a naive majority-class guess: {reported_accuracy - base_rate:.3f}")
print()

# Methodology question 2: does pooling a position tier across content types hide thin cells?
# (illustrative check on my own lane's data, not FlyRank's literal warehouse split)
floor = df[df["impressions_90d"] >= 100]
pooled = floor.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("Pooled CTR by position tier (all content types together):")
print(pooled)
print()

split_cells = floor.groupby(["position_tier", "content_type"])["ctr"].agg(["mean", "count"])
thin_cells = split_cells[split_cells["count"] < 30].sort_values("count")
print(f"Position-tier x content-type cells with n < 30, out of {len(split_cells)} total cells: {len(thin_cells)}")
print(thin_cells)


Implied base rate for the majority class ('up'): 0.621
Reported holdout accuracy: 0.71
Real skill above a naive majority-class guess: 0.089

Pooled CTR by position tier (all content types together):
                   mean  count
position_tier                 
page_1         0.354760   8633
top_3          0.334128    533
striking       0.255782   5903
page_3_5       0.142359   6058
deep           0.055415    879

Position-tier x content-type cells with n < 30, out of 14 total cells: 3
                                   mean  count
position_tier content_type                    
top_3         comparison article  0.000      1
              feedly article      2.900      5
deep          feedly article      0.044     10


## 2. My model under an honest split (before/after)

Same Week-5 model, same candidate pool, same honest (no-CTR) feature set, same label
(`is_underperforming`). Two splits: a plain random 80/25 split (no grouping, the "before"), and
the client-grouped split from Week 5 (the "after"). Client overlap between train and test is
printed for each, this is the number that explains the gap.


In [3]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

visible = df["impressions_90d"] >= 500
in_range = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
lane = df[visible & in_range].copy()
tier_median_ctr = lane.groupby("position_tier")["ctr"].transform("median")
lane["ctr_gap"] = lane["ctr"] - tier_median_ctr
lane["is_underperforming"] = (lane["ctr_gap"] < 0).astype(int)

numeric_honest = ["impressions_90d", "avg_position", "word_count", "content_age_days",
                   "days_since_last_update", "engagement_rate", "scroll_rate", "ai_traffic_pct",
                   "search_volume", "competition", "cpc"]
categorical_honest = ["content_type", "main_intent", "position_tier"]

model_df = lane.copy()
for c in numeric_honest:
    model_df[c] = model_df[c].fillna(model_df[c].median())
for c in categorical_honest:
    model_df[c] = model_df[c].fillna("unknown")

X = model_df[numeric_honest + categorical_honest]
y = model_df["is_underperforming"]
groups = model_df["client_id"]

pre = ColumnTransformer([
    ("num", StandardScaler(), numeric_honest),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_honest),
])

def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)
    return np.asarray(y_true)[order[:k]].mean()

def run_split(name, train_idx, test_idx):
    pipe = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
    pipe.fit(X.iloc[train_idx], y.iloc[train_idx])
    proba = pipe.predict_proba(X.iloc[test_idx])[:, 1]
    y_te = y.iloc[test_idx]
    return {"split": name, "n_test": len(test_idx),
            "roc_auc": round(roc_auc_score(y_te, proba), 3),
            "precision_at_20": round(precision_at_k(y_te.values, proba, 20), 3),
            "precision_at_50": round(precision_at_k(y_te.values, proba, 50), 3),
            "client_overlap": len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))}

rand_train_idx, rand_test_idx = train_test_split(np.arange(len(X)), test_size=0.25, random_state=42, stratify=y)
before = run_split("random (before)", rand_train_idx, rand_test_idx)

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
grp_train_idx, grp_test_idx = next(gss.split(X, y, groups))
after = run_split("client-grouped (after)", grp_train_idx, grp_test_idx)

comparison = pd.DataFrame([before, after])
comparison


,split,n_test,roc_auc,precision_at_20,precision_at_50,client_overlap
0,random (before),3006,0.743,1.0,0.94,27
1,client-grouped (after),824,0.727,0.7,0.68,0


**Reading the gap:** the random split leaks 27 of 28 clients across train and test, and
precision@20 hits a perfect 1.000, a strong sign the model is partly recognizing "this is client
X's content" rather than a generalizable content pattern. The client-grouped split (0 overlap,
same as Week 5) drops precision@20 to 0.700 and precision@50 to 0.680, a meaningful fall from
0.94, that gap is itself the finding: roughly a quarter of the random split's apparent skill was
memorization, not signal that would transfer to a brand new client. The grouped number is the
one that should be trusted and reported going forward.


## 3. Leakage audit

The attack checklist from the hunting-leakage-and-validating skill, run against the final
feature set from Week 5.


In [4]:
LEAKY_COLUMNS = {"ctr", "ctr_gap", "clicks_90d", "trend_direction", "trend_pct"}
FEATURE_COLUMNS = set(numeric_honest) | set(categorical_honest)

print("[check] No label-derived or sibling columns in the features:")
print("        overlap =", LEAKY_COLUMNS & FEATURE_COLUMNS, "(empty set = pass)")
print()

print("[check] No product flags / existing-system scores as features:")
print("        this starter dataset ships no product flags (health_score, priority_score,")
print("        action_type) at all, so there is nothing to accidentally include, confirmed by")
print("        checking the column list directly:")
print("       ", [c for c in df.columns if "score" in c.lower() or "flag" in c.lower() or "priority" in c.lower()] or "none found")
print()

print("[check] Population selection checked for outcome-window information:")
print("        candidate pool = impressions_90d >= 500 and 0 < avg_position <= 20, both from")
print("        the SAME 90-day window as every feature, no forward-looking filter applied.")
print()

print("[check] Split grouped by the repeating entity:")
print("        client-grouped split used in section 2, 0 client overlap confirmed.")
print()

print("[check] Base rate printed next to every metric:")
print(f"        is_underperforming base rate (full candidate pool): {lane['is_underperforming'].mean():.3f}")


[check] No label-derived or sibling columns in the features:
        overlap = set() (empty set = pass)

[check] No product flags / existing-system scores as features:
        this starter dataset ships no product flags (health_score, priority_score,
        action_type) at all, so there is nothing to accidentally include, confirmed by
        checking the column list directly:
        none found

[check] Population selection checked for outcome-window information:
        candidate pool = impressions_90d >= 500 and 0 < avg_position <= 20, both from
        the SAME 90-day window as every feature, no forward-looking filter applied.

[check] Split grouped by the repeating entity:
        client-grouped split used in section 2, 0 client overlap confirmed.

[check] Base rate printed next to every metric:
        is_underperforming base rate (full candidate pool): 0.490


In [5]:
# The confession test: train once WITH the suspect feature (ctr_gap), once WITHOUT.
# A collapse from ~1.0 back down to the honest number is the leakage signature.
numeric_leaky = numeric_honest + ["ctr_gap"]
X_leaky = model_df[numeric_leaky + categorical_honest]
pre_leaky = ColumnTransformer([
    ("num", StandardScaler(), numeric_leaky),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_honest),
])

pipe_with = Pipeline([("pre", pre_leaky), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
pipe_with.fit(X_leaky.iloc[grp_train_idx], y.iloc[grp_train_idx])
auc_with = roc_auc_score(y.iloc[grp_test_idx], pipe_with.predict_proba(X_leaky.iloc[grp_test_idx])[:, 1])

pipe_without = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
pipe_without.fit(X.iloc[grp_train_idx], y.iloc[grp_train_idx])
auc_without = roc_auc_score(y.iloc[grp_test_idx], pipe_without.predict_proba(X.iloc[grp_test_idx])[:, 1])

print(f"[check] Top feature importance sanity-checked, WITH ctr_gap:    ROC AUC = {auc_with:.3f}")
print(f"[check] Top feature importance sanity-checked, WITHOUT ctr_gap: ROC AUC = {auc_without:.3f}")
print(f"        Collapse of {auc_with - auc_without:.3f} on removal confirms ctr_gap was the leak,")
print("        exactly the pattern the skill describes, this is why it stays out of the honest model.")
print()
print("[check] Metrics recomputed out-of-fold, never in-sample: all numbers above are test-set only.")
print("[check] Sealed/holdout claims: not applicable, no sealed evaluation is claimed in this project.")


[check] Top feature importance sanity-checked, WITH ctr_gap:    ROC AUC = 1.000
[check] Top feature importance sanity-checked, WITHOUT ctr_gap: ROC AUC = 0.727
        Collapse of 0.273 on removal confirms ctr_gap was the leak,
        exactly the pattern the skill describes, this is why it stays out of the honest model.

[check] Metrics recomputed out-of-fold, never in-sample: all numbers above are test-set only.
[check] Sealed/holdout claims: not applicable, no sealed evaluation is claimed in this project.


## 4. Claim rewrite

**My boldest sentence, from Week 2's `w02_ml_task_framing.ipynb`:**

> "The pattern (expected CTR depends on position, and the gap from that expectation is what
> matters) is real but has enough structure that a position-tier-adjusted score does
> meaningfully better than an unadjusted rule, worth checking with data rather than assuming."

At the time that was written, "does meaningfully better" hadn't actually been measured, the only
evidence available then was the naive-rule flag-rate check (a global `ctr < 0.2` threshold
flagging 44% of `page_1` pages versus 91% of `deep` pages). That's a real, useful number, but it
shows the naive rule is *inconsistent* across tiers, it doesn't show the tier-adjusted score is
*better* in any measured sense. Claiming "meaningfully better" got ahead of what had actually
been checked.

**Rewritten in safe language:**

> "A single global CTR threshold produces inconsistent flag rates across position tiers (44% of
> `page_1` pages flagged versus 91% of `deep` pages, using `ctr < 0.2` as an illustrative cutoff),
> an observed asymmetry that motivates a position-tier-adjusted score as a directional
> improvement. Whether the adjusted score is actually better at surfacing genuine review
> candidates is a decision-support question, not yet formally validated here against an
> unadjusted rule using precision@K on held-out data."


In [6]:
# The evidence the rewritten claim is actually built on, recomputed here for the record.
# Same candidate pool w02 used (impressions_90d >= 100, avg_position > 0), not the tighter
# w04/w05 pool, so this reproduces the exact 44% / 91% figures quoted in the rewritten claim.
w02_lane = df[(df["impressions_90d"] >= 100) & (df["avg_position"] > 0)].copy()
naive_flag_rate = w02_lane.groupby("position_tier").apply(lambda g: (g["ctr"] < 0.2).mean())
print("Share of each tier flagged by a naive global rule (ctr < 0.2), the actual evidence behind")
print("the rewritten claim above, nothing stronger than this was measured at the time:")
print(naive_flag_rate.sort_values(ascending=False))


Share of each tier flagged by a naive global rule (ctr < 0.2), the actual evidence behind
the rewritten claim above, nothing stronger than this was measured at the time:
position_tier
deep        0.908987
page_3_5    0.753219
striking    0.573098
top_3       0.510319
page_1      0.443646
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.